# Overall preparation of datasets
- Subsetting of adminstrative bounds and merging with census population data
- Conversion to geospatial and subsetting of building csv file
- Subsetting of OSM datasets

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from openpyxl import load_workbook
import os

## Getting boundaries of NCR (and, additionally, Rizal)

In [2]:
adm1 = gpd.read_file("..//data//adm_bounds//phl_admbnda_adm1_psa_namria_20231106.shp")
ncr = adm1.loc[adm1["ADM1_EN"]=="National Capital Region (NCR)"]
adm2 = gpd.read_file("..//data//adm_bounds//phl_admbnda_adm2_psa_namria_20231106.shp")
rzl = adm2.loc[adm2["ADM2_EN"]=="Rizal"]

del adm1, adm2

print(ncr),print(rzl)

                          ADM1_EN ADM1_PCODE            ADM0_EN ADM0_PCODE  \
12  National Capital Region (NCR)       PH13  Philippines (the)         PH   

         date    validOn validTo  Shape_Leng  Shape_Area ADM1ALT1EN  \
12 2022-11-09 2023-11-06     NaT    2.303584    0.050207        NCR   

     AREA_SQKM                                           geometry  
12  598.539573  POLYGON ((121.09951 14.76921, 121.09951 14.769...  
   ADM2_EN ADM2_PCODE                   ADM1_EN ADM1_PCODE            ADM0_EN  \
20   Rizal    PH04058  Region IV-A (Calabarzon)       PH04  Philippines (the)   

   ADM0_PCODE       date    validOn validTo  Shape_Leng  Shape_Area  \
20         PH 2022-11-09 2023-11-06     NaT    2.982169    0.100696   

   ADM2ALT1EN    AREA_SQKM                                           geometry  
20        NaN  1200.370955  MULTIPOLYGON (((121.45105 14.59522, 121.44111 ...  


(None, None)

## Subsetting L3/L4 admin bounds and joining with census population
### Some caveats
There are changes in the administrative boundaries in the census data that are not yet updated for the admin bounds shapefile used.

The outdated admin bounds will be used as the standard.

The changes that are not yet reflected on the admin bounds (and, consequently, in this study) are the following:
- Transfer of some former Makati barangays to Taguig
- Division of the former Barangay 176 in Caloocan to multiple new barangays

### NCR

In [3]:
city_shp = gpd.read_file("..//data//adm_bounds//phl_admbnda_adm3_psa_namria_20231106.shp")
ncr_city_shp = city_shp.loc[city_shp["ADM1_EN"]=='National Capital Region (NCR)'].reset_index(drop=True)[list(city_shp.columns[:8]) + ["geometry"]]

wb = load_workbook("..//data//census//NCR.xlsx")
city_list = wb.sheetnames[1:]

brgy_shp = gpd.read_file("..//data//adm_bounds//phl_admbnda_adm4_psa_namria_20231106.shp")
ncr_brgy_shp = brgy_shp.loc[brgy_shp["ADM1_EN"]=='National Capital Region (NCR)'].reset_index(drop=True)[list(brgy_shp.columns[:11]) + ["geometry"]]
ncr_brgy_shp["city_fmt"] = ncr_brgy_shp["ADM3_EN"].str.replace("City of ","").str.replace(" City","")
ncr_brgy_shp["brgy_fmt"] = ncr_brgy_shp["ADM4_EN"].str.split(" \\(",expand=True)[0].str.lower().str.strip()

C:\Users\Ronilo\miniconda3\envs\geo\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [4]:
temp_df = []
for cty in city_list:
    num_space = 0
    row = 7
    while num_space < 2:
        if wb[cty][f"B{row}"].value is None:
            num_space +=1
        elif num_space == 1:
            num_space = 0
        else:
            temp_df.append([wb[cty][f"B{row}"].value,cty,wb[cty][f"D{row}"].value])
        row +=1

brgy_pop_df = pd.DataFrame(temp_df,columns=["brgy","city","pop"])
brgy_pop_df["city_fmt"] = brgy_pop_df["city"].str.replace("City of ","").str.replace(" City","")
brgy_pop_df.loc[(brgy_pop_df["city_fmt"]=="Caloocan")|(brgy_pop_df["city_fmt"]=="Taguig"),"brgy"] = brgy_pop_df.loc[(brgy_pop_df["city_fmt"]=="Caloocan")|(brgy_pop_df["city_fmt"]=="Taguig"),"brgy"].replace("(?<!Barangay) 1$","",regex=True)
brgy_pop_df["brgy_fmt"] = brgy_pop_df["brgy"].str.split(" \\(",expand=True)[0].str.lower().str.strip()

In [5]:
ncr_brgy_fin = ncr_brgy_shp.merge(brgy_pop_df,how="left",on=["city_fmt","brgy_fmt"])
missing_brgy = ncr_brgy_fin.loc[pd.isna(ncr_brgy_fin["pop"])]

ncr_brgy_temp = ncr_brgy_shp.merge(brgy_pop_df,how="right",on=["city_fmt","brgy_fmt"])[["ADM4_EN","city_fmt","brgy_fmt","brgy","city","pop"]]
missing_brgy2 = ncr_brgy_temp.loc[pd.isna(ncr_brgy_temp["ADM4_EN"])]

for x in missing_brgy.itertuples():
    if x.city_fmt == "Makati":
        ncr_brgy_fin.loc[ncr_brgy_fin.index==x.Index,"pop"] = missing_brgy2.loc[(missing_brgy2.city_fmt=="Taguig")&(missing_brgy2.brgy_fmt==x.brgy_fmt),"pop"].values[0]
    elif x.ADM4_EN == "North Bay Blvd., North":
        ncr_brgy_fin.loc[ncr_brgy_fin.index==x.Index,"pop"] = missing_brgy2.loc[missing_brgy2.brgy=="North Bay Boulevard North","pop"].values[0]
    elif x.ADM4_EN == "Barangay 176":
        ncr_brgy_fin.loc[ncr_brgy_fin.index==x.Index,"pop"] = np.sum(missing_brgy2.loc[missing_brgy2.brgy.str.contains("Barangay 176"),"pop"].values)

ncr_brgy_fin.drop(columns=["city_fmt","brgy_fmt","brgy","city"],inplace=True)

ncr_city_fin = ncr_city_shp.merge(ncr_brgy_fin[["ADM3_EN","pop"]].groupby("ADM3_EN",as_index=False).sum(),on="ADM3_EN")

ncr_city_fin["pop"] = ncr_city_fin["pop"].astype('Int64')
ncr_brgy_fin["pop"] = ncr_brgy_fin["pop"].astype('Int64')

In [6]:
ncr_city_fin.to_file("..//data//ncr//adm_bounds.gpkg",driver="GPKG",layer="city")
ncr_brgy_fin.to_file("..//data//ncr//adm_bounds.gpkg",driver="GPKG",layer="barangay")

In [7]:
del ncr_city_fin, ncr_brgy_fin, ncr_brgy_temp, missing_brgy, missing_brgy2, brgy_pop_df, temp_df, wb, city_list, ncr_city_shp, ncr_brgy_shp

### Rizal

In [8]:
ws = load_workbook("..//data//census//CALABARZON.xlsx")["Rizal"]

muni_temp = []
brgy_temp = []
num_space = 0
row = 7
muni = ""
while num_space < 2:
    if ws[f"B{row}"].value is None:
        num_space +=1
    elif num_space == 1:
        muni_temp.append([ws[f"B{row}"].value,ws[f"D{row}"].value])
        muni = ws[f"B{row}"].value
        num_space = 0
    else:
        brgy_temp.append([ws[f"B{row}"].value,muni,ws[f"D{row}"].value])
    row +=1

muni_pop_df = pd.DataFrame(muni_temp,columns=["muni","pop"])
muni_pop_df["muni_fmt"] = muni_pop_df["muni"].str.lower().str.strip()

brgy_pop_df = pd.DataFrame(brgy_temp,columns=["brgy","muni","pop"])
brgy_pop_df["muni_fmt"] = brgy_pop_df["muni"].str.lower().str.strip()
brgy_pop_df["brgy_fmt"] = brgy_pop_df["brgy"].str.lower().str.strip()

In [9]:
rzl_city_shp = city_shp.loc[city_shp["ADM2_EN"]=='Rizal'].reset_index(drop=True)[list(city_shp.columns[:8]) + ["geometry"]]
rzl_city_shp["muni_fmt"] = rzl_city_shp["ADM3_EN"].str.split(" \\(",expand=True)[0].str.lower().str.strip()
rzl_city_fin = rzl_city_shp.merge(muni_pop_df,on="muni_fmt",how="left").drop(columns=["muni_fmt","muni"])

rzl_brgy_shp = brgy_shp.loc[brgy_shp["ADM2_EN"]=='Rizal'].reset_index(drop=True)[list(brgy_shp.columns[:11]) + ["geometry"]]
rzl_brgy_shp["muni_fmt"] = rzl_brgy_shp["ADM3_EN"].str.split(" \\(",expand=True)[0].str.lower().str.strip()
rzl_brgy_shp["brgy_fmt"] = rzl_brgy_shp["ADM4_EN"].str.split(" \\(",expand=True)[0].str.lower().str.strip()
rzl_brgy_fin = rzl_brgy_shp.merge(brgy_pop_df,on=["muni_fmt","brgy_fmt"],how="left").drop(columns=["muni_fmt","muni","brgy_fmt","brgy"])

In [10]:
rzl_city_fin.to_file("..//data//rizal//adm_bounds.gpkg",driver="GPKG",layer="city")
rzl_brgy_fin.to_file("..//data//rizal//adm_bounds.gpkg",driver="GPKG",layer="barangay")

In [11]:
del rzl_brgy_fin, rzl_city_fin, rzl_brgy_shp, rzl_city_shp, brgy_pop_df, muni_pop_df, brgy_temp, muni_temp, ws, city_shp, brgy_shp

## Conversion to geospatial and subsetting of building csv file

In [12]:
bldg_csv = pd.read_csv("..//data//bldg_raw//339_buildings.csv")

bldg_shp = gpd.GeoDataFrame(data=bldg_csv[["latitude","longitude","area_in_meters","confidence","full_plus_code"]],
                           geometry=gpd.GeoSeries.from_wkt(bldg_csv["geometry"],crs="EPSG:4326"),crs="EPSG:4326")

del bldg_csv

bldg_ncr = bldg_shp.loc[bldg_shp.intersects(ncr.iloc[0].geometry)].drop_duplicates(subset=["full_plus_code"]).reset_index(drop=True)
print("Number of buildings in dataset (NCR):",len(bldg_ncr))
bldg_ncr.to_file("..//data//ncr//buildings.gpkg", driver="GPKG", layer="bldg")
del bldg_ncr

bldg_rzl = bldg_shp.loc[bldg_shp.intersects(rzl.iloc[0].geometry)].drop_duplicates(subset=["full_plus_code"]).reset_index(drop=True)
print("Number of buildings in dataset (Rizal):",len(bldg_rzl))
bldg_rzl.to_file("..//data//rizal//buildings.gpkg", driver="GPKG", layer="bldg")
del bldg_rzl

Number of buildings in dataset (NCR): 2244868
Number of buildings in dataset (Rizal): 819532


In [13]:
del bldg_shp

## Subsetting of OSM datasets
OSM datasets used:
- buildings
- points of interest (points and polygons)
- roads

In [15]:
osm_filenames = list(set([x.split(".")[0] for x in os.listdir("..//data//osm_raw")]))
for x in osm_filenames:
    osm_shp = gpd.read_file(f"..//data//osm_raw//{x}.shp")
    
    osm_shp_ncr = osm_shp.loc[osm_shp.intersects(ncr.iloc[0].geometry)].reset_index(drop=True)
    osm_shp_ncr.to_file("..//data//ncr//OSM.gpkg", driver="GPKG", layer=x)
    
    osm_shp_rzl = osm_shp.loc[osm_shp.intersects(rzl.iloc[0].geometry)].reset_index(drop=True)
    osm_shp_rzl.to_file("..//data//rizal//OSM.gpkg", driver="GPKG", layer=x)
    
    print(x)
    print("Number of features (NCR):",len(osm_shp_ncr))
    print("Number of features (Rizal):",len(osm_shp_rzl))

gis_osm_pois_free_1
Number of features (NCR): 37114
Number of features (Rizal): 3879
gis_osm_buildings_a_free_1
Number of features (NCR): 1155623
Number of features (Rizal): 308582
gis_osm_roads_free_1
Number of features (NCR): 156769
Number of features (Rizal): 44193
gis_osm_pois_a_free_1
Number of features (NCR): 11769
Number of features (Rizal): 2898


In [16]:
del osm_shp, osm_shp_ncr, osm_shp_rzl